In [ ]:
from langchain_openai import ChatOpenAI

In [5]:
import sys
import uuid

# uuid_utils ko override karke standard Python uuid library use karwana
sys.modules['uuid_utils'] = uuid
sys.modules['uuid_utils.compat'] = uuid

In [6]:
from typing import TypedDict
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END

In [ ]:
load_dotenv()

llm = ChatOpenAI()

In [7]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [8]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [9]:
def generate_explanation(state: JokeState):

    prompot = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompot).content

    return {'explanation': response}

In [10]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic': 'pizza'}, config=config1)

# Time Travelling

In [ ]:
workflow.get_state({'configurable': {"thread_id": "1", "checkpoint_id": "12eafefn7q8hgydt6dn"}})

## Resume from checkpoint

In [ ]:
workflow.invoke(None, {'configurable': {"thread_id": "1", "checkpoint_id": "12eafefn7q8hgydt6dn"}})

## Updating State

In [ ]:
workflow.update_state({{'configurable': {"thread_id": "1", "checkpoint_id": "12eafefn7q8hgydt6dn"}}}, {'topic': 'samosa'})